# Counting Worlds - Demo with Sample Data

**Language:** English | [日本語版はこちら](demo_ja.ipynb)

This notebook demonstrates the core functionality of the "Counting Worlds" project, which analyzes historical Nigerian newspaper data from the late 19th and early 20th centuries.

## Overview

This demonstration covers:

1. **Unified Data Loading Function**: A single function that can load different types of newspaper data (editorials and reader correspondence) with automatic format detection
2. **Data Structure Comparison**: Understanding the differences between editorial and correspondence data formats
3. **Basic Statistical Analysis**: Computing metrics like article counts, text lengths, and temporal distributions

## Data Sources

The project works with data from two major Nigerian newspapers:
- **Lagos Observer (LOE)**: Editorials and reader correspondence (LOC)
- **Lagos Weekly Record (LWR/LWRE)**: Editorials

For this demo, we use simplified sample data located in the `sample_data/` directory.

## 1. Import Required Libraries

**Purpose**: This cell imports the Python libraries needed for data analysis.

**What each library does**:
- `pandas`: Main library for data manipulation and analysis
- `numpy`: Numerical computing library
- `os`: File and path operations
- `re`: Regular expressions for text processing
- `datetime`: Date and time handling

In [1]:
import pandas as pd
import numpy as np
import os
import re
from datetime import datetime

print("Libraries imported successfully")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

Libraries imported successfully
pandas version: 2.3.3
numpy version: 2.3.5


## 2. Define Data Loading Functions

**Purpose**: This cell defines the core functions for loading and preprocessing newspaper data.

### Main Function: `load_newspaper_data()`

This function provides a unified interface for loading different types of newspaper data:

**Key Features**:
1. **Automatic Type Detection**: Recognizes whether data is editorial or correspondence based on filename
2. **Automatic Source Detection**: Identifies the newspaper source (Lagos Observer, Lagos Weekly Record, etc.)
3. **Column Standardization**: Maps various column names to a unified schema
4. **Metadata Addition**: Adds `data_source` and `article_type` columns
5. **Special Handling**: Different logic for correspondence data (handles composite IDs like "1_1", "1_2")

**Parameters**:
- `filepath`: Path to the CSV file
- `data_type`: Optional, specify 'editorial' or 'correspondence' (auto-detected if None)
- `data_source`: Optional, specify source name (auto-detected if None)

### Helper Function: `preprocess_text()`

Cleans text data by:
- Converting to lowercase
- Removing non-alphabetic characters
- Normalizing whitespace

In [2]:
def load_newspaper_data(filepath, data_type=None, data_source=None):
    """
    Unified Nigerian newspaper data loading function (improved version)
    
    Parameters:
    - filepath: CSV file path
    - data_type: 'editorial', 'correspondence', or None (auto-detect)
    - data_source: Data source name, or None (auto-detect)
    
    Returns:
    - DataFrame with unified columns and metadata
    """
    import pandas as pd
    import os
    
    # Load the CSV file
    df = pd.read_csv(filepath, encoding='utf-8')
    
    filename = os.path.basename(filepath).lower()
    
    # Auto-detect data type from filename if not specified
    if data_type is None:
        if 'loe' in filename:
            data_type = 'editorial'
        elif 'loc' in filename:
            data_type = 'correspondence'
        elif 'lwr' in filename or 'lwre' in filename:
            data_type = 'editorial'
        elif 'editorial' in filename:
            data_type = 'editorial'
        elif 'correspondence' in filename or 'letter' in filename:
            data_type = 'correspondence'
        else:
            data_type = 'editorial'  # Default value
    
    # Auto-detect data source if not specified
    if data_source is None:
        if 'loe' in filename or 'loc' in filename:
            data_source = 'Lagos Observer'
        elif 'lwr' in filename or 'lwre' in filename:
            data_source = 'Lagos Weekly Record'
        elif 'lagos_observer' in filename or 'lo_' in filename:
            data_source = 'Lagos Observer'
        elif 'weekly_record' in filename or 'wr_' in filename:
            data_source = 'Lagos Weekly Record'
        else:
            # Extract a meaningful name from filepath
            basename = os.path.splitext(os.path.basename(filepath))[0]
            # Clean up the name
            clean_name = basename.replace('_', ' ').title()
            data_source = f'Custom Source ({clean_name})'
    
    # Add metadata columns
    df['data_source'] = data_source
    df['article_type'] = data_type
    
    # Handle LOC-specific column mapping first
    if data_type == 'correspondence':
        # LOC-specific mapping adjustment
        if 'no' in df.columns and 'id_1' in df.columns:
            df['id'] = df['no']  # Serial number to id
            df['composite_id'] = df['id_1']  # Save composite ID (1_1 format) to separate column
    
    # Standard column mapping for all data types
    column_mapping = {
        # Text columns
        'Text': 'text', 'TEXT': 'text',
        
        # Date columns - including Publication Date for LOE/LWRE
        'Date': 'date', 'DATE': 'date',
        'Publication Date': 'date', 'publication date': 'date',
        
        # Year columns
        'Year': 'year', 'YEAR': 'year',
        
        # ID columns (for LOE/LWRE, not LOC)
        'ID': 'id', 'Id': 'id', 'Article_ID': 'id', 'article_id': 'id'
    }
    
    for old_col, new_col in column_mapping.items():
        if old_col in df.columns:
            df.rename(columns={old_col: new_col}, inplace=True)
    
    # Ensure essential columns exist
    if 'id' not in df.columns:
        df['id'] = range(1, len(df) + 1)
    
    if 'year' not in df.columns and 'date' in df.columns:
        try:
            df['year'] = pd.to_datetime(df['date']).dt.year
        except:
            df['year'] = None
    
    return df


def preprocess_text(text):
    """
    Helper function to preprocess text data
    """
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = ' '.join(text.split())
    return text

## 3. Load Editorial Data

**Purpose**: Load and inspect sample editorial data.

**What this cell does**:
1. Loads editorial data from `sample_data/sample_editorial.csv`
2. Displays basic information about the loaded data:
   - Number of records
   - Detected data source
   - Article type
   - Number of columns

**Expected Output**: Information confirming successful loading of 5 editorial records.

In [3]:
print("=== Loading Editorial Data ===")
editorial_df = load_newspaper_data('./sample_data/sample_editorial.csv')

print(f"Loading complete: {len(editorial_df)} records")
print(f"Data source: {editorial_df.data_source.iloc[0]}")
print(f"Article type: {editorial_df.article_type.iloc[0]}")
print(f"Number of columns: {len(editorial_df.columns)}")

=== Loading Editorial Data ===
Loading complete: 5 records
Data source: Custom Source (Sample Editorial)
Article type: editorial
Number of columns: 7


## 4. Load Correspondence Data

**Purpose**: Load and inspect sample reader correspondence data.

**What this cell does**:
1. Loads correspondence (reader letters) data from `sample_data/sample_correspondence.csv`
2. Displays basic information including:
   - Number of records
   - Detected data source
   - Article type
   - Number of columns
   - Examples of composite IDs (special ID format for correspondence)

**Note**: Correspondence data has a special ID format (e.g., "1_1", "1_2") where:
- First number: The issue/date ID
- Second number: The letter number within that issue

**Expected Output**: Information confirming successful loading of 5 correspondence records.

In [4]:
print("=== Loading Correspondence Data ===")
correspondence_df = load_newspaper_data('./sample_data/sample_correspondence.csv')

print(f"Loading complete: {len(correspondence_df)} records")
print(f"Data source: {correspondence_df.data_source.iloc[0]}")
print(f"Article type: {correspondence_df.article_type.iloc[0]}")
print(f"Number of columns: {len(correspondence_df.columns)}")
print(f"Composite ID examples: {correspondence_df.composite_id.tolist()[:3]}")

=== Loading Correspondence Data ===
Loading complete: 5 records
Data source: Custom Source (Sample Correspondence)
Article type: correspondence
Number of columns: 9
Composite ID examples: ['1_1', '1_2', '1_3']


## 5. Compare Data Structures

**Purpose**: Examine and compare the column structures of editorial vs. correspondence data.

**What this cell does**:
1. Lists all columns in editorial data
2. Lists all columns in correspondence data
3. Displays sample rows from each dataset

**Key Differences to Observe**:

**Editorial columns**:
- `id`: Unique identifier
- `text`: Article text content
- `date`: Publication date
- `year`: Year extracted from date
- `Years`: Decade (e.g., "1880s")
- `data_source`: Newspaper name
- `article_type`: "editorial"

**Correspondence columns**:
- `no`: Original serial number
- `id_1`: Composite ID (issue_letter format)
- `text`: Letter text content
- `year`, `date`: Temporal information
- `data_source`, `article_type`: Metadata
- `id`: Converted from `no`
- `composite_id`: Preserved from `id_1`

**Why This Matters**: Understanding these structural differences helps when writing analysis code that works with both data types.

In [5]:
print("=== Data Structure Comparison ===")
print("\n【Editorial Data Column Structure】")
for i, col in enumerate(editorial_df.columns, 1):
    print(f"{i:2d}. {col}")

print("\n【Correspondence Data Column Structure】")
for i, col in enumerate(correspondence_df.columns, 1):
    print(f"{i:2d}. {col}")

print("\n【Editorial Data Sample】")
print(editorial_df.head())

print("\n【Correspondence Data Sample】")
print(correspondence_df.head())

=== Data Structure Comparison ===

【Editorial Data Column Structure】
 1. id
 2. text
 3. date
 4. year
 5. Years
 6. data_source
 7. article_type

【Correspondence Data Column Structure】
 1. no
 2. id_1
 3. text
 4. year
 5. date
 6. data_source
 7. article_type
 8. id
 9. composite_id

【Editorial Data Sample】
   id                                               text        date  year  \
0   1  The government has announced new regulations f...  1882/03/02  1882   
1   2  Education remains a priority for the colonial ...  1882/03/09  1882   
2   3  The railway construction project continues to ...  1882/03/16  1882   
3   4  Local merchants express concerns about new tax...  1882/03/23  1882   
4   5  The Governor addressed the council on matters ...  1882/03/30  1882   

   Years                       data_source article_type  
0  1880s  Custom Source (Sample Editorial)    editorial  
1  1880s  Custom Source (Sample Editorial)    editorial  
2  1880s  Custom Source (Sample Editorial)    

## 6. Basic Statistical Analysis

**Purpose**: Compute and display basic statistics about the loaded datasets.

**What this cell does**:

1. **Dataset Overview**:
   - Counts editorials and correspondence separately
   - Reports total article count

2. **Temporal Information**:
   - Shows year range for editorials
   - Shows year range for correspondence
   - Displays decade information if available

3. **Text Length Analysis**:
   - Computes character counts for each article
   - Calculates average text length by type
   - Helps understand content volume differences

4. **Aggregated Statistics by Article Type**:
   - Groups all data by article type
   - Shows count, mean, and standard deviation of text length
   - Shows temporal range (min/max year)

**Use Cases**: These statistics help researchers understand:
- Dataset composition and balance
- Temporal coverage
- Typical article/letter lengths
- Variability in content volume

In [6]:
print("=== Basic Statistics ===")

print("\n【Dataset Overview】")
print(f"Number of editorials: {len(editorial_df)}")
print(f"Number of correspondence: {len(correspondence_df)}")
print(f"Total articles: {len(editorial_df) + len(correspondence_df)}")

print("\n【Year Information】")
print(f"Editorial years: {editorial_df.year.unique()}")
print(f"Editorial decades: {editorial_df.Years.unique() if 'Years' in editorial_df.columns else 'N/A'}")
print(f"Correspondence years: {correspondence_df.year.unique()}")

print("\n【Text Length Statistics】")
editorial_df['text_length'] = editorial_df['text'].str.len()
correspondence_df['text_length'] = correspondence_df['text'].str.len()

print(f"Average editorial length: {editorial_df.text_length.mean():.1f} characters")
print(f"Average correspondence length: {correspondence_df.text_length.mean():.1f} characters")

print("\n【Statistics by Article Type】")
combined_df = pd.concat([editorial_df, correspondence_df], ignore_index=True)
print(combined_df.groupby('article_type').agg({
    'text_length': ['count', 'mean', 'std'],
    'year': ['min', 'max']
}).round(1))

=== Basic Statistics ===

【Dataset Overview】
Number of editorials: 5
Number of correspondence: 5
Total articles: 10

【Year Information】
Editorial years: [1882]
Editorial decades: ['1880s']
Correspondence years: [1882]

【Text Length Statistics】
Average editorial length: 63.6 characters
Average correspondence length: 58.6 characters

【Statistics by Article Type】
               text_length             year      
                     count  mean  std   min   max
article_type                                     
correspondence           5  58.6  6.7  1882  1882
editorial                5  63.6  4.2  1882  1882


## Conclusion

This demo has shown:

1. ✅ How to load different types of newspaper data using a unified function
2. ✅ The structural differences between editorial and correspondence data
3. ✅ Basic statistical analysis techniques for newspaper corpora

## Next Steps

For more advanced analysis, see the main analysis notebook:
- `Counting_Worlds_cleaned.ipynb`: Full analysis with visualization and text mining techniques

## Questions or Issues?

Please refer to the project README or contact the project maintainers.